# State AI Policy: source validation

## Summary

The 18 sampled records match LegiScan bill identities. Seven bills have an intermediate API stage in a closed session, but **all seven already have a source status of `Failed - Adjourned`**. Their official histories support retaining that status.

Four sampled records originally marked Pending have later signing or chaptering evidence. These are dated validation observations, not replacement training labels. The original and prepared datasets remain unchanged by this notebook.

## Context and method

This diagnostic sample contains six records each from California, Colorado and Texas. Selection first sought three Pending records per state, then filled six using terminal statuses and a deterministic hash order. Texas contributes six terminal-status records. This convenience sample does not estimate a national error rate.

The October 13 observations are preserved as recorded, including their access and coverage limits. The October 19 LegiScan comparison adds matched state, bill and session identifiers, provider stages and history dates. Official state histories and session-closure sources were rechecked October 19 for the seven closed-session cases.

LegiScan's `Introduced`, `Engrossed` and `Passed` labels describe provider stages. A stage, a session closure and an enacted outcome are different facts. The checks below keep them separate. No API key, network access or data regeneration is needed to run this notebook.

## Data

### 1. Load the frozen source and validation tables

Run from the repository root or `notebooks/`. `source_row` is the one-based position in the unchanged raw CSV, not a model feature.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if not (project_root / "data/raw/Final_Database.csv").exists():
    project_root = project_root.parent

raw_path = project_root / "data/raw/Final_Database.csv"
raw = pd.read_csv(raw_path, dtype=str, keep_default_na=False, skip_blank_lines=False)
validation_dir = project_root / "data/validation"
observations = pd.read_csv(validation_dir / "bill_status_observations.csv", dtype=str, keep_default_na=False)
comparison = pd.read_csv(validation_dir / "legiscan_comparison.csv", dtype=str, keep_default_na=False)
closed = pd.read_csv(validation_dir / "closed_session_review.csv", dtype=str, keep_default_na=False)

### 2. Confirm source linkage and matched identities

The checksum identifies the published snapshot. Matching the source fields prevents a reordered or different CSV from silently changing the records under review.

In [2]:
source_md5 = hashlib.md5(raw_path.read_bytes()).hexdigest()
assert source_md5 == "397a13c53e719e64e52725e7995fd8ca"
assert raw.shape == (2571, 29)
assert len(comparison) == 18 and comparison["source_row"].is_unique
assert comparison["source_md5"].eq(source_md5).all()
assert comparison["identity_matched"].eq("true").all()
assert comparison["api_bill_id"].is_unique

source_columns = ["Jurisdiction", "Bill_Number", "Bill_Title", "Introduction_Date", "Version_Date", "Bill_Status"]
positions = observations["source_row"].astype(int) - 1
pd.testing.assert_frame_equal(
    raw.loc[positions, source_columns].reset_index(drop=True),
    observations[source_columns].reset_index(drop=True),
)
assert comparison.set_index("source_row")["source_status"].to_dict() == observations.set_index("source_row")["Bill_Status"].to_dict()
display(comparison.groupby("state").agg(Records=("source_row", "size"), Matched=("identity_matched", lambda values: values.eq("true").sum())))

,Records,Matched
state,,
CA,6,6
CO,6,6
TX,6,6


## Results

### 3. Compare source statuses with API stages

Read this as a comparison of different fields and check dates. A different stage label alone does not establish an incorrect source outcome. The CSV retains exact bill/session identifiers, public links and API check times.

In [3]:
display(pd.crosstab(
    comparison["source_status"], comparison["api_stage"],
    rownames=["Source status"], colnames=["API stage"],
))

API stage,Engrossed,Failed,Introduced,Passed,Vetoed
Source status,,,,,
Enacted - Act No. 2025-341,0,0,0,1,0
Enacted - Act No. 2025-466,0,0,0,1,0
Enacted - Act No. 2025-673,0,0,0,1,0
Failed - Adjourned,2,1,5,0,0
Failed - Adjourned - Postponed Indefinitely,0,1,0,0,0
Pending,0,0,1,4,1


### 4. Review the seven closed-session cases

Colorado HB25B-1008 belongs to the 2025 extraordinary session, not a regular-session bill with the same number. The official page records Lost; the session adjourned August 26, 2025.

Texas's 89th Regular Session ended June 2, 2025. The six cited histories show incomplete bills. The official Legislative Reference Library explains that unfinished bills die at session end. For Texas, `Failed at session close` is an inference from those histories and that rule, rather than a literal API label.

All seven source records already say `Failed - Adjourned`. Retain the original baseline statuses. Keep API stages and closure evidence in the validation tables; no correction or exclusion is supported by this stage distinction alone. A latest action date is not a failure date.

In [4]:
assert len(closed) == 7
assert closed["source_status"].eq("Failed - Adjourned").all()
assert set(closed["source_row"]) == set(comparison.loc[
    comparison["session_sine_die"].eq("1") & comparison["api_stage"].isin(["Introduced", "Engrossed", "Enrolled"]),
    "source_row",
])
display(closed[["state", "bill", "source_status", "api_stage", "last_action_date", "session_closed_date"]].rename(columns={
    "source_status": "Source status", "api_stage": "API stage",
    "last_action_date": "Last action", "session_closed_date": "Session closed",
}))

,state,bill,Source status,API stage,Last action,Session closed
0,CO,HB25B-1008,Failed - Adjourned,Introduced,2025-08-25,2025-08-26
1,TX,HB641,Failed - Adjourned,Introduced,2025-03-04,2025-06-02
2,TX,SB747,Failed - Adjourned,Engrossed,2025-05-26,2025-06-02
3,TX,SB1298,Failed - Adjourned,Introduced,2025-02-28,2025-06-02
4,TX,HB4448,Failed - Adjourned,Introduced,2025-04-02,2025-06-02
5,TX,SB1822,Failed - Adjourned,Engrossed,2025-05-07,2025-06-02
6,TX,HB1405,Failed - Adjourned,Introduced,2025-05-14,2025-06-02


### 5. Keep four later enactment observations separate

The following source-Pending records have explicit signing or chaptering history evidence. LegiScan code 4 is named `Passed`; the action evidence supplies the additional context. These differences were already observed in the October 13 check and are confirmed by the October 19 API histories.

An updated-label experiment would need a documented cutoff and separate results. It must not introduce later information into features intended to describe an earlier prediction point.

In [5]:
pending_with_evidence = comparison.loc[
    comparison["source_status"].eq("Pending") & comparison["signing_or_chaptering_evidence"].eq("True"),
    ["state", "bill", "source_status", "api_stage", "api_last_action_date", "api_last_action"],
]
assert len(pending_with_evidence) == 4
with pd.option_context("display.max_colwidth", 75):
    display(pending_with_evidence.rename(columns={"api_last_action_date": "Action date", "api_last_action": "Latest history action"}))

,state,bill,source_status,api_stage,Action date,Latest history action
0,CA,SB241,Pending,Passed,2025-10-01,"Chaptered by Secretary of State. Chapter 214, Statutes of 2025."
1,CA,SB951,Pending,Passed,2026-09-30,"Chaptered by Secretary of State. Chapter 860, Statutes of 2026."
7,CO,HB1195,Pending,Passed,2026-06-03,Governor Signed
8,CO,HB1263,Pending,Passed,2026-05-29,Governor Signed


### 6. Preserve the unresolved cases and date definitions

- **California AB2291:** the available history records a canceled hearing. That does not establish a terminal outcome. The API reports `sine_die=0`; no independently refreshed session calendar is used to decide the outcome here.
- **Colorado HB26-1210:** API Vetoed agrees with the earlier veto-history observation. The October 13 official-page Signed Act/summary wording conflict remains documented in `bill_status_observations.csv`.
- **Texas HB4448:** the source short title differs from the official school-library caption. The raw text header identifies Hickland's HB4448, supporting identity. Full-text/version equivalence has not been established; the title difference alone does not justify exclusion.

Three Texas source introduction dates differ from API first-history dates, which record filing. Preserve both definitions rather than overwriting the source introduction dates.

In [6]:
source_dates = pd.to_datetime(observations["Introduction_Date"], format="%m/%d/%Y").dt.strftime("%Y-%m-%d")
date_review = comparison.merge(
    observations[["source_row", "Introduction_Date"]].assign(source_date_iso=source_dates),
    on="source_row", validate="one_to_one",
)
date_review = date_review.loc[date_review["source_date_iso"].ne(date_review["api_first_history_date"])]
display(date_review[["state", "bill", "Introduction_Date", "api_first_history_date"]].rename(columns={
    "Introduction_Date": "Source introduction", "api_first_history_date": "API first history",
}))

,state,bill,Source introduction,API first history
12,TX,HB641,01/14/2025,2024-11-12
13,TX,SB747,01/14/2025,2025-01-08
17,TX,HB1405,01/14/2025,2024-11-19


### 7. Link 20 source texts to their version references

`source_text_review.csv` compares the frozen CSV with the cached version references checked October 20. These 20 selected records cover the earlier outcome/sponsor cases plus additional version checks. They are not a random sample. The table preserves exact source statuses and dates, identifies the API bill and session, and links to the available introduced document. Reference-retrieval times remain in `text_version_review.csv`; the new table records the comparison time.

The header label and date are separate checks. Three Colorado introduced dates come from official version tables because the API dates are placeholders. Delaware SCR18 has an undated Draft reference, so its introduced date and introduced-document URL remain blank. An API version reference does not establish that the source text is identical to that version.

In [7]:
source_text_review = pd.read_csv(
    validation_dir / "source_text_review.csv", dtype=str, keep_default_na=False,
)
version_references = pd.read_csv(
    validation_dir / "text_version_review.csv", dtype=str, keep_default_na=False,
)
assert len(source_text_review) == 20 and source_text_review["source_row"].is_unique
assert source_text_review["source_md5"].eq(source_md5).all()
assert set(source_text_review["source_row"]) == set(version_references["source_row"])
identity_columns = ["api_bill_id", "api_session_id", "state", "bill_number"]
pd.testing.assert_frame_equal(
    source_text_review.set_index("source_row")[identity_columns].sort_index(),
    version_references.set_index("source_row")[identity_columns].sort_index(),
)
source_positions = source_text_review["source_row"].astype(int) - 1
assert source_positions.between(0, len(raw) - 1).all()
assert source_text_review["source_status"].tolist() == raw.loc[source_positions, "Bill_Status"].tolist()
assert source_text_review["source_version_date_raw"].tolist() == raw.loc[source_positions, "Version_Date"].tolist()
parsed_source_dates = pd.to_datetime(
    source_text_review["source_version_date_raw"], format="%m/%d/%Y",
).dt.strftime("%Y-%m-%d")
assert parsed_source_dates.equals(source_text_review["source_version_date_iso"])
assert source_text_review["full_text_identity_not_established"].eq("True").all()
source_text_review["later_stage_header"] = source_text_review["later_stage_header"].eq("True")

### 8. Screen all raw bill-text headers

This conservative screen covers **2,571 raw records**, rather than the 2,567 records retained after shared duplicate removal. It looks only for explicit stage labels near the start of `Bill_Text`, before the first digest, summary or body marker. It also accepts California's formal Legislative Counsel's Digest bill-number label ending in “as introduced”, before the bill body. “Introduced by” names an author and is not a version label.

The screen finds 252 explicit later-stage headers, 31 explicit introduced headers and 2,288 without an explicit label under these rules. `NOT_EXPLICIT` means unknown, not introduced. Header labels do not establish when every clause, summary or sponsor feature became available. The screen is a review aid, not a historical-eligibility test.

In [8]:
import re


def header_stage(text):
    header = re.split(
        r"LEGISLATIVE COUNSEL'S DIGEST|Bill Summary|WHEREAS,|BE IT ENACTED",
        text, maxsplit=1, flags=re.I,
    )[0][:1800]
    lines = [line.strip() for line in header.splitlines() if line.strip()]
    for line in lines:
        if re.fullmatch(r"PREAMENDED|REENGROSSED|ENGROSSED|INTRODUCED|ENROLLED", line):
            return line
    if any(re.match(r"AMENDED IN (ASSEMBLY|SENATE)\b", line) for line in lines):
        return "AMENDED"
    if any(re.fullmatch(r"CHAPTER \d+", line) for line in lines):
        return "CHAPTERED"
    before_body = re.split(
        r"Bill Summary|WHEREAS,|BE IT ENACTED", text, maxsplit=1, flags=re.I,
    )[0][:1800]
    digest_label = re.search(
        r"LEGISLATIVE COUNSEL'S DIGEST\s+"
        r"(?:AB|SB|ACR|SCR|AJR|SJR|ACA|SCA|HR|SR)\s*\d+,\s+as introduced\b",
        before_body, flags=re.I,
    )
    if digest_label:
        return "INTRODUCED"
    return "NOT_EXPLICIT"


assert header_stage("A BILL\nBE IT ENACTED\nThis section quotes a bill as introduced.") == "NOT_EXPLICIT"
assert header_stage("INTRODUCED\nA BILL FOR AN ACT") == "INTRODUCED"


header_screen = pd.DataFrame({
    "source_row": range(1, len(raw) + 1),
    "screened_header_stage": raw["Bill_Text"].map(header_stage),
})
header_screen["source_row"] = header_screen["source_row"].astype(str)
header_counts = header_screen["screened_header_stage"].value_counts().rename_axis("Header stage")
display(header_counts.to_frame("Raw records"))
assert header_counts.sum() == 2571

,Raw records
Header stage,
NOT_EXPLICIT,2288
CHAPTERED,101
AMENDED,99
INTRODUCED,31
ENROLLED,27
ENGROSSED,16
REENGROSSED,6
PREAMENDED,3


### 9. Review the flagged sample and compare dates

All 20 saved stage labels must match the screen of the raw headers. Nine sampled records contain explicit later-stage labels: five source-Pending records, three source-Passed records and one source-Failed record, using the publisher's grouped statuses. The table below also preserves each exact source status. These flags do not establish nine incorrect outcomes or nine cases of target leakage.

Twelve source version dates follow the introduced reference, seven match and one lacks a verified introduced date. Colorado HB26-1195 shows why dates alone are insufficient: its source version date matches the introduced reference, but its text is labeled `PREAMENDED`. The official introduced PDF is labeled `INTRODUCED`.

In [9]:
sample_screen = source_text_review.merge(header_screen, on="source_row", validate="one_to_one")
assert sample_screen["source_header_stage"].equals(sample_screen["screened_header_stage"])
later_stages = {"PREAMENDED", "REENGROSSED", "ENGROSSED", "AMENDED", "CHAPTERED", "ENROLLED"}
assert sample_screen["later_stage_header"].equals(sample_screen["screened_header_stage"].isin(later_stages))
flagged = sample_screen.loc[sample_screen["later_stage_header"]].copy()
flagged["source_grouped_status"] = raw.loc[
    flagged["source_row"].astype(int) - 1, "Bill_Status_Grouped",
].to_numpy()
assert len(flagged) == 9
assert flagged["source_grouped_status"].value_counts().to_dict() == {"Pending": 5, "Passed": 3, "Failed": 1}
with pd.option_context("display.max_colwidth", 55):
    display(flagged[[
        "source_row", "state", "bill_number", "source_status", "source_header_stage",
        "source_version_date_iso", "introduced_reference_date_iso",
    ]])

,source_row,state,bill_number,source_status,source_header_stage,source_version_date_iso,introduced_reference_date_iso
0,55,CA,SB241,Pending,AMENDED,2025-06-26,2025-01-30
8,1586,CA,SB951,Pending,AMENDED,2026-04-09,2026-02-02
9,1613,CA,AB2291,Pending,AMENDED,2026-04-06,2026-02-19
11,1628,CA,SB361,Enacted - Act No. 2025-466,CHAPTERED,2025-10-08,2025-02-13
12,1631,CA,AB578,Enacted - Act No. 2025-341,CHAPTERED,2025-10-06,2025-02-12
13,1632,CA,AB621,Enacted - Act No. 2025-673,CHAPTERED,2025-10-13,2025-02-13
14,1637,CO,HB1195,Pending,PREAMENDED,2026-02-11,2026-02-11
15,1638,CO,HB1210,Pending,REENGROSSED,2026-03-26,2026-02-13
17,1644,CO,HB1008,Failed - Adjourned,ENGROSSED,2025-08-24,2025-08-21


In [10]:
source_version_dates = pd.to_datetime(source_text_review["source_version_date_iso"], format="%Y-%m-%d")
introduced_dates = pd.to_datetime(
    source_text_review["introduced_reference_date_iso"], format="%Y-%m-%d", errors="coerce",
)
date_comparison = pd.Series("Introduced date unavailable", index=source_text_review.index)
date_comparison.loc[source_version_dates.gt(introduced_dates)] = "After introduced reference"
date_comparison.loc[source_version_dates.eq(introduced_dates)] = "Matches introduced reference"
date_comparison.loc[source_version_dates.lt(introduced_dates)] = "Before introduced reference"
display(date_comparison.value_counts().rename_axis("Source version date").to_frame("Sample records"))
assert date_comparison.value_counts().to_dict() == {
    "After introduced reference": 12,
    "Matches introduced reference": 7,
    "Introduced date unavailable": 1,
}
display(source_text_review.loc[source_text_review["source_row"].eq("1637"), [
    "state", "bill_number", "source_version_date_iso", "introduced_reference_date_iso",
    "source_header_stage", "introduced_document_url",
]])

,Sample records
Source version date,
After introduced reference,12
Matches introduced reference,7
Introduced date unavailable,1


,state,bill_number,source_version_date_iso,introduced_reference_date_iso,source_header_stage,introduced_document_url
14,CO,HB1195,2026-02-11,2026-02-11,PREAMENDED,https://leg.colorado.gov/bill_files/111810/dow...


Only three cached introduced PDFs received full-body comparisons. Their normalized token sequences differ from the source texts, including the one with an `INTRODUCED` header. Differences can include formatting, extraction and content; no similarity threshold establishes version identity. The remaining records do not have full-body comparisons. The CSV records these limits row by row.

Keep these findings separate from modeling inputs. Dropping only the nine sampled flags would leave other known later-stage headers. Dropping all detected later-stage headers would still leave 2,288 unknown headers and unverified summary/sponsor timing, and could change the outcome mix. Neither filter establishes a clean introduction-time cohort. A sensitivity comparison is an analysis-owner decision and must compare input variants on the same rows and splits.

## Takeaways

Keep the frozen recorded outcomes for the primary retrospective baseline. The seven closed-session records support the existing Failed/Adjourned labels. Later observations stay in separate validation tables. The source snapshot does not establish a uniform outcome observation date; publication, bill-version and API-check dates are different dates. Pending remains unresolved rather than becoming Failed merely because it has not passed.

The nine selected later-stage headers and the full-corpus screen document a timing limitation. They do not justify an automatic filter or establish an introduction-time prediction dataset. Our descriptive analysis of NCSL summaries does not need to drop the nine bills on the basis of their bill-text headers. Using summaries or topics for prediction needs its own timing and training/test checks.

The supervised analysis owns target definitions, eligibility, splits and any optional matched-cohort sensitivity. Record the feature-availability rule and outcome observation rule before a label-refresh experiment. Compare feature variants on identical rows and splits. Topic analysis retains its own descriptive text and method decisions.

These checks establish sample linkage and specific findings, not full-corpus text equivalence or historical eligibility. This notebook reads files and displays results; it does not change raw/prepared data, export model features, train models or call the API.

## Sources

- Bhargava and Soliman, [U.S. State AI Legislation Corpus v4](https://zenodo.org/records/20781731), `Final_Database_10.csv`. The repository copy is unchanged.
- [LegiScan](https://legiscan.com/) API bill metadata/history, retrieved October 19, 2026. Selected fields are reorganized and renamed in `legiscan_comparison.csv`; stage labels follow the [API manual, Status/Progress section](https://api.legiscan.com/dl/LegiScan_API_User_Manual.pdf). LegiScan data is licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
- Official state bill-history links and per-record check times appear in the validation CSVs. The seven-bill review also cites the [Texas post-session FAQ](https://www.lrl.texas.gov/whatsNew/client/index.cfm/2025/6/3/Whats-Next-PostSession-FAQ-and-Bill-Statistics-89th-RS) and [Colorado extraordinary-session journal](https://content.leg.colorado.gov/sites/default/files/2025_house_journal.pdf), printed page 76 (PDF page 80).